In [1]:
import pandas as pd
import numpy as np

def analyze_customer_leak(file_path, time_col='Hourly', consumption_col='Consumption m3'):
    """
    Ingests any customer water consumption dataset, builds dual baselines,
    evaluates leakage criteria, and prints a conclusive diagnostic report.
    """
    try:
        # --- 1. Load and Standardize Data ---
        df = pd.read_excel(file_path, header=2)
        df[time_col] = pd.to_datetime(df[time_col])
        df = df.sort_values(time_col).reset_index(drop=True)
        
        # Extract core time features
        df['hour'] = df[time_col].dt.hour
        df['day_of_week'] = df[time_col].dt.dayofweek
        df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
        
        # Split into historical window (first month) vs recent window (last 4 weeks)
        max_date = df[time_col].max()
        min_date = df[time_col].min()
        four_weeks_ago = max_date - pd.Timedelta(weeks=4)
        first_month_end = min_date + pd.Timedelta(days=30)
        
        df['period'] = np.where(df[time_col] >= four_weeks_ago, 'Recent', 'Middle')
        df.loc[df[time_col] <= first_month_end, 'period'] = 'Historical'
        
        # --- 2. Calculate Recent Baseline Profile ---
        recent_data = df[df['period'] == 'Recent']
        recent_profile = recent_data.groupby(['is_weekend', 'hour'])[consumption_col].median().reset_index()
        recent_profile.rename(columns={consumption_col: 'baseline_median'}, inplace=True)
        
        recent_std = recent_data.groupby(['is_weekend', 'hour'])[consumption_col].std().reset_index()
        recent_std.rename(columns={consumption_col: 'baseline_std'}, inplace=True)
        
        # Merge baselines back
        df = pd.merge(df, recent_profile, on=['is_weekend', 'hour'], how='left')
        df = pd.merge(df, recent_std, on=['is_weekend', 'hour'], how='left')
        
        # --- 3. Calculate Deviations ---
        df['abs_deviation'] = df[consumption_col] - df['baseline_median']
        df['z_score'] = df['abs_deviation'] / (df['baseline_std'] + 1e-5)
        
        # --- 4. Evaluate Automation Logic ---
        leak_detected = False
        leak_reasons = []
        
        # CRITERION A: Slow/Pre-Existing Leak Check (Night Flow Shift)
        # Baseline lowest low from the first month (using 5th percentile to block anomalies)
        hist_night = df[(df['period'] == 'Historical') & (df['hour'].isin([1, 2, 3, 4]))]
        historical_min_flow = hist_night[consumption_col].quantile(0.05) if not hist_night.empty else 0
        
        # Lowest low during the most recent week
        recent_week_night = df[(df[time_col] >= (max_date - pd.Timedelta(days=7))) & (df['hour'].isin([1, 2, 3, 4]))]
        recent_min_flow = recent_week_night[consumption_col].min() if not recent_week_night.empty else 0
        
        # If the minimum night flow has drifted up significantly (using a buffer of 0.2 m3 or 30% increase)
        night_flow_drift = recent_min_flow - historical_min_flow
        if night_flow_drift > 0.3 and recent_min_flow > (historical_min_flow * 1.3):
            leak_detected = True
            leak_reasons.append(f"Slow/Pre-existing Leak: Night minimum flow crept from {historical_min_flow:.3f} m3 up to {recent_min_flow:.3f} m3.")
            
        # CRITERION B: Sudden Burst Leak Check (Consecutive high Z-scores)
        # Create a boolean mask where Z-score is anomalously high
        df['is_anomaly'] = (df['period'] == 'Recent') & (df['z_score'] > 3.5)
        
        # Calculate consecutive anomalies using pandas rolling window
        consecutive_anomalies = df['is_anomaly'].astype(int).rolling(window=3).sum()
        if (consecutive_anomalies >= 3).any():
            leak_detected = True
            max_z = df.loc[df['period'] == 'Recent', 'z_score'].max()
            leak_reasons.append(f"Sudden Burst Leak: Detected high consumption spikes lasting 3+ consecutive hours (Peak Z-Score: {max_z:.1f}).")
            
        # --- 5. Print Execution Dashboard ---
        print("="*60)
        print(f"AUTOMATED LEAK DETECTION REPORT FOR: {file_path}")
        print(f"Data Timeline: {min_date.strftime('%Y-%m-%d')} to {max_date.strftime('%Y-%m-%d')}")
        print("="*60)
        
        if leak_detected:
            print("🔴 STATUS: LEAK SUSPECTED")
            print("\nFlags Triggered:")
            for reason in leak_reasons:
                print(f"  - {reason}")
        else:
            print("🟢 STATUS: NORMAL (NO LEAK DETECTED)")
            print("\nFlags Triggered:\n  - None. Water profiles match historical baseline behaviors cleanly.")
        print("="*60 + "\n")
        
        return df # Returns processed dataframe if you want to inspect rows further
        
    except Exception as e:
        print(f"❌ Error processing file {file_path}: {str(e)}")
        return None

# =====================================================================
# HOW TO RUN IT:
# Just change the string filename below to point to any customer's file.
# =====================================================================
#processed_df = analyze_customer_leak("water_data.xlsx")


In [ ]:
import os

# Loop through a whole folder of files
data_folder = "./datasets/"
for file_name in os.listdir(data_folder):
    if file_name.endswith(".xlsx"):
        full_path = os.path.join(data_folder, file_name)
        analyze_customer_leak(full_path)


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def plot_leak_diagnostic(file_path, time_col='Hourly', consumption_col='Consumption m3'):
    """
    Loads a specific customer file, runs the leak math, and plots a detailed
    diagnostic chart showing why the leak flag was triggered.
    """
    # 1. Run the data prep steps exactly like the main script
    df = pd.read_excel(file_path, header=2)
    df[time_col] = pd.to_datetime(df[time_col])
    df = df.sort_values(time_col).reset_index(drop=True)
    
    df['hour'] = df[time_col].dt.hour
    df['day_of_week'] = df[time_col].dt.dayofweek
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
    
    max_date = df[time_col].max()
    min_date = df[time_col].min()
    four_weeks_ago = max_date - pd.Timedelta(weeks=4)
    first_month_end = min_date + pd.Timedelta(days=30)
    
    df['period'] = np.where(df[time_col] >= four_weeks_ago, 'Recent', 'Middle')
    df.loc[df[time_col] <= first_month_end, 'period'] = 'Historical'
    
    # 2. Build Profiles
    recent_data = df[df['period'] == 'Recent']
    recent_profile = recent_data.groupby(['is_weekend', 'hour'])[consumption_col].median().reset_index()
    recent_profile.rename(columns={consumption_col: 'baseline_median'}, inplace=True)
    
    recent_std = recent_data.groupby(['is_weekend', 'hour'])[consumption_col].std().reset_index()
    recent_std.rename(columns={consumption_col: 'baseline_std'}, inplace=True)
    
    df = pd.merge(df, recent_profile, on=['is_weekend', 'hour'], how='left')
    df = pd.merge(df, recent_std, on=['is_weekend', 'hour'], how='left')
    
    df['abs_deviation'] = df[consumption_col] - df['baseline_median']
    df['z_score'] = df['abs_deviation'] / (df['baseline_std'] + 1e-5)
    df['is_anomaly'] = (df['period'] == 'Recent') & (df['z_score'] > 3.5)
    
    # 3. Filter for the final 14 days to keep the visual clean and zoomed in
    last_14_days = df[df[time_col] >= (max_date - pd.Timedelta(days=14))].copy()
    
    # 4. Generate Plot Canvas
    plt.figure(figsize=(16, 7))
    sns.set_theme(style="whitegrid")
    
    # Line 1: Actual Consumption
    sns.lineplot(data=last_14_days, x=time_col, y=consumption_col, 
                 label='Actual Consumption', color='#1f77b4', linewidth=2.5, zorder=2)
    
    # Line 2: Expected Baseline
    sns.lineplot(data=last_14_days, x=time_col, y='baseline_median', 
                 label='Expected 4-Week Baseline', color='#ff7f0e', linestyle='--', linewidth=2, zorder=2)
    
    # Line 3: Historical Night Minimum benchmark
    hist_night = df[(df['period'] == 'Historical') & (df['hour'].isin([1, 2, 3, 4]))]
    historical_min_flow = hist_night[consumption_col].quantile(0.05) if not hist_night.empty else 0
    plt.axhline(y=historical_min_flow, color='red', linestyle=':', linewidth=2, 
                label=f'Historical Night Low Benchmark ({historical_min_flow:.3f} m3)')
    
    # Highlight burst zones (points where Z-Score > 3.5)
    anomalies = last_14_days[last_14_days['is_anomaly'] == True]
    if not anomalies.empty:
        plt.scatter(anomalies[time_col], anomalies[consumption_col], 
                    color='crimson', edgecolor='black', s=60, label='Burst Anomaly (Z > 3.5)', zorder=4)
        
    # 5. Styling and Labels
    plt.title(f'Diagnostic Leak Profile: {file_path}', fontsize=14, fontweight='bold', pad=15)
    plt.xlabel('Date & Time', fontsize=12, labelpad=10)
    plt.ylabel('Water Volume (m3)', fontsize=12, labelpad=10)
    plt.xticks(rotation=45)
    plt.legend(loc='upper left', frameon=True, facecolor='white')
    plt.tight_layout()
    plt.show()




In [ ]:
import os

# Loop through a whole folder of files
data_folder = "./datasets/"
for file_name in os.listdir(data_folder):
    if file_name.endswith(".xlsx"):
        full_path = os.path.join(data_folder, file_name)
        plot_leak_diagnostic(full_path)


In [ ]:
import os
import pandas as pd
import numpy as np

def generate_leak_summary_tables(folder_path, time_col='Hourly', consumption_col='Consumption m3'):
    """
    Loops through all Excel files in a folder and prints the high-deviation
    breakdown tables with Z-scores for every single customer.
    """
    # Verify folder exists
    if not os.path.exists(folder_path):
        print(f"❌ Error: The folder path '{folder_path}' does not exist.")
        return

    # Get all excel files in the folder
    excel_files = [f for f in os.listdir(folder_path) if f.endswith(('.xlsx', '.xls'))]
    
    if not excel_files:
        print(f"⚠️ No Excel files found in the folder '{folder_path}'.")
        return

    print(f"Found {len(excel_files)} customer files to process.\n")

    for file_name in excel_files:
        file_path = os.path.join(folder_path, file_name)
        
        try:
            # --- 1. Load and Clean Data ---
            df = pd.read_excel(file_path, header=2)
            df[time_col] = pd.to_datetime(df[time_col])
            df = df.sort_values(time_col).reset_index(drop=True)
            
            # --- 2. Feature Engineering ---
            df['hour'] = df[time_col].dt.hour
            df['day_of_week'] = df[time_col].dt.dayofweek
            df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
            
            # Identify timeline window (Last 4 weeks = Recent)
            max_date = df[time_col].max()
            four_weeks_ago = max_date - pd.Timedelta(weeks=4)
            df['period'] = np.where(df[time_col] >= four_weeks_ago, 'Recent', 'Historical')
            
            # --- 3. Build Baseline Profiles ---
            recent_data = df[df['period'] == 'Recent']
            
            recent_profile = recent_data.groupby(['is_weekend', 'hour'])[consumption_col].median().reset_index()
            recent_profile.rename(columns={consumption_col: 'recent_baseline_m3'}, inplace=True)
            
            recent_std = recent_data.groupby(['is_weekend', 'hour'])[consumption_col].std().reset_index()
            recent_std.rename(columns={consumption_col: 'baseline_std_m3'}, inplace=True)
            
            # Merge profiles back into main dataframe
            df = pd.merge(df, recent_profile, on=['is_weekend', 'hour'], how='left')
            df = pd.merge(df, recent_std, on=['is_weekend', 'hour'], how='left')
            
            # --- 4. Deviation & Z-Score Math ---
            df['abs_deviation_m3'] = df[consumption_col] - df['recent_baseline_m3']
            df['z_score_deviation'] = df['abs_deviation_m3'] / (df['baseline_std_m3'] + 1e-5)
            
            # Fill missing historical rows with zeroes
            df[['abs_deviation_m3', 'z_score_deviation']] = df[['abs_deviation_m3', 'z_score_deviation']].fillna(0)
            
            # --- 5. Isolate Top 10 Anomalies ---
            recent_df = df[df['period'] == 'Recent'].copy()
            top_deviations = recent_df.sort_values(by='abs_deviation_m3', ascending=False).head(10)
            
            # Format table structure
            summary_table = pd.DataFrame({
                'Timestamp': top_deviations[time_col].dt.strftime('%Y-%m-%d %H:%M'),
                'Actual Consumption': top_deviations[consumption_col].round(4),
                'Expected Baseline': top_deviations['recent_baseline_m3'].round(4),
                'Extra Water Wasted (Abs Dev)': top_deviations['abs_deviation_m3'].round(4),
                'Z-Score (Severity)': top_deviations['z_score_deviation'].round(1)
            })
            
            # --- 6. Print Header and Table ---
            print("=" * 80)
            print(f"DEVIATION PROFILE TABLE FOR CUSTOMER SHEET: {file_name}")
            print("=" * 80)
            
            # Use to_string to ensure it formats reliably in any setup
            print(summary_table.to_string(index=False))
            print("\n" + "-"*80 + "\n")
            
        except Exception as e:
            print(f"❌ Error processing file {file_name}: {str(e)}")
            print("\n" + "-"*80 + "\n")

# =====================================================================
# RUN THE LOOP
# Point this to the folder path where your 7 Excel files are saved.
# Replace '.' with your path, e.g., r"C:\Users\yourname\Documents\WaterData"
# =====================================================================
generate_leak_summary_tables(folder_path="./datasets")


In [8]:
import pandas as pd
import numpy as np

def analyze_customer_leak(file_path, time_col='Hourly', consumption_col='Consumption m3'):
    """
    Ingests any customer water consumption dataset, builds dual baselines grouped by 
    the exact day of the week, evaluates leakage criteria, and prints a diagnostic report.
    """
    try:
        # --- 1. Load and Standardize Data ---
        df = pd.read_excel(file_path, header=2)
        df[time_col] = pd.to_datetime(df[time_col])
        df = df.sort_values(time_col).reset_index(drop=True)
        
        # Extract core time features
        df['hour'] = df[time_col].dt.hour
        df['day_of_week'] = df[time_col].dt.dayofweek # 0=Monday, 4=Friday, 6=Sunday
        
        # Split into historical window (first month) vs recent window (last 4 weeks)
        max_date = df[time_col].max()
        min_date = df[time_col].min()
        four_weeks_ago = max_date - pd.Timedelta(weeks=4)
        first_month_end = min_date + pd.Timedelta(days=30)
        
        df['period'] = np.where(df[time_col] >= four_weeks_ago, 'Recent', 'Middle')
        df.loc[df[time_col] <= first_month_end, 'period'] = 'Historical'
        
        # --- 2. Calculate Recent Baseline Profile (FIXED: Grouping by exact day_of_week) ---
        recent_data = df[df['period'] == 'Recent']
        recent_profile = recent_data.groupby(['day_of_week', 'hour'])[consumption_col].median().reset_index()
        recent_profile.rename(columns={consumption_col: 'baseline_median'}, inplace=True)
        
        recent_std = recent_data.groupby(['day_of_week', 'hour'])[consumption_col].std().reset_index()
        recent_std.rename(columns={consumption_col: 'baseline_std'}, inplace=True)
        
        # Merge baselines back using the exact day_of_week match
        df = pd.merge(df, recent_profile, on=['day_of_week', 'hour'], how='left')
        df = pd.merge(df, recent_std, on=['day_of_week', 'hour'], how='left')
        
        # --- 3. Calculate Deviations ---
        df['abs_deviation'] = df[consumption_col] - df['baseline_median']
        df['z_score'] = df['abs_deviation'] / (df['baseline_std'] + 1e-5)
        
        # --- 4. Evaluate Automation Logic ---
        leak_detected = False
        leak_reasons = []
        
        # CRITERION A: Slow/Pre-Existing Leak Check (Night Flow Shift)
        hist_night = df[(df['period'] == 'Historical') & (df['hour'].isin([1, 2, 3, 4]))]
        historical_min_flow = hist_night[consumption_col].quantile(0.05) if not hist_night.empty else 0
        
        recent_week_night = df[(df[time_col] >= (max_date - pd.Timedelta(days=7))) & (df['hour'].isin([1, 2, 3, 4]))]
        recent_min_flow = recent_week_night[consumption_col].min() if not recent_week_night.empty else 0
        
        night_flow_drift = recent_min_flow - historical_min_flow
        if night_flow_drift > 0.3 and recent_min_flow > (historical_min_flow * 1.3):
            leak_detected = True
            leak_reasons.append(f"Slow/Pre-existing Leak: Night minimum flow crept from {historical_min_flow:.3f} m3 up to {recent_min_flow:.3f} m3.")
            
        # CRITERION B: Sudden Burst Leak Check (Consecutive high Z-scores)
        df['is_anomaly'] = (df['period'] == 'Recent') & (df['z_score'] > 3.5)
        
        consecutive_anomalies = df['is_anomaly'].astype(int).rolling(window=3).sum()
        if (consecutive_anomalies >= 3).any():
            leak_detected = True
            max_z = df.loc[df['period'] == 'Recent', 'z_score'].max()
            leak_reasons.append(f"Sudden Burst Leak: Detected high consumption spikes lasting 3+ consecutive hours (Peak Z-Score: {max_z:.1f}).")
            
        # --- 5. Print Execution Dashboard ---
        print("="*60)
        print(f"AUTOMATED LEAK DETECTION REPORT FOR: {file_path}")
        print(f"Data Timeline: {min_date.strftime('%Y-%m-%d')} to {max_date.strftime('%Y-%m-%d')}")
        print("="*60)
        
        if leak_detected:
            print("🔴 STATUS: LEAK SUSPECTED")
            print("\nFlags Triggered:")
            for reason in leak_reasons:
                print(f"  - {reason}")
        else:
            print("🟢 STATUS: NORMAL (NO LEAK DETECTED)")
            print("\nFlags Triggered:\n  - None. Water profiles match historical baseline behaviors cleanly.")
        print("="*60 + "\n")
        
        # --- 6. Print Updated Top 10 Anomalies Table ---
        recent_df = df[df['period'] == 'Recent'].copy()
        top_deviations = recent_df.sort_values(by='abs_deviation', ascending=False).head(10)
        
        summary_table = pd.DataFrame({
            'Timestamp': top_deviations[time_col].dt.strftime('%Y-%m-%d %H:%M'),
            'Actual Consumption': top_deviations[consumption_col].round(4),
            'Expected Baseline': top_deviations['baseline_median'].round(4),
            'Extra Water Wasted (Abs Dev)': top_deviations['abs_deviation'].round(4),
            'Z-Score (Severity)': top_deviations['z_score'].round(1)
        })
        print("📊 TOP 10 HIGHEST DEVIATION EVENTS FOR THIS USER:")
        print(summary_table.to_string(index=False))
        print("="*60 + "\n")
        
        return df 
        
    except Exception as e:
        print(f"❌ Error processing file {file_path}: {str(e)}")
        return None


In [ ]:
import os

# Loop through a whole folder of files
data_folder = "./datasets/"
for file_name in os.listdir(data_folder):
    if file_name.endswith(".xlsx"):
        full_path = os.path.join(data_folder, file_name)
        analyze_customer_leak(full_path)
